# Capítulo 2: Análisis de Estacionariedad

## 2.1 Marco teórico

La estacionariedad es condición necesaria para la validez del modelo ARIMA. Una serie $\{y_t\}$ es **estacionaria en covarianza** si:

$$E[y_t] = \mu \quad \forall t$$
$$\text{Var}(y_t) = \sigma^2 < \infty \quad \forall t$$
$$\text{Cov}(y_t, y_{t+k}) = \gamma_k \quad \text{depende solo de } k$$

Se aplican dos tests con hipótesis complementarias:

| Test | $H_0$ | $H_1$ | Rechazar $H_0$ si |
|------|--------|--------|-------------------|
| **ADF** | Raíz unitaria (no estacionaria) | Estacionaria | p-valor $< \alpha$ |
| **KPSS** | Estacionaria | No estacionaria | p-valor $< \alpha$ |

La **convergencia de ambos tests** proporciona evidencia robusta.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import yfinance as yf
from datetime import datetime

# Recargar datos
raw = yf.download('BTC-USD', start='2014-09-17', end=datetime.today().strftime('%Y-%m-%d'), progress=False)
btc = raw[['Close']].copy()
btc.columns = ['Price']
btc.index = pd.DatetimeIndex(btc.index).normalize()
btc.dropna(inplace=True)
btc['LogReturn'] = np.log(btc.Price / btc.Price.shift(1))

def test_estacionariedad(serie, nombre, alpha=0.05):
    s   = serie.dropna()
    sep = "-" * 65

    adf_stat, adf_p, adf_lags, _, adf_cv, _ = adfuller(s, autolag='AIC')
    adf_concl = '[OK] Estacionaria' if adf_p < alpha else '[X]  No estacionaria'

    kpss_stat, kpss_p, kpss_lags, kpss_cv = kpss(s, regression='c', nlags='auto')
    kpss_concl = '[OK] Estacionaria' if kpss_p > alpha else '[X]  No estacionaria'

    print("\n" + sep)
    print(f"  Serie : {nombre}")
    print(sep)
    print(f"  ADF  | Estadistico = {adf_stat:9.4f} | p = {adf_p:.4f} | Lags = {adf_lags:2d} | {adf_concl}")
    print(f"  KPSS | Estadistico = {kpss_stat:9.4f} | p = {kpss_p:.4f} | Lags = {kpss_lags:2d} | {kpss_concl}")
    print(f"  Valores criticos ADF  (1%, 5%, 10%)       : {[round(v,3) for v in adf_cv.values()]}")
    print(f"  Valores criticos KPSS (10%, 5%, 2.5%, 1%) : {[round(v,3) for v in kpss_cv.values()]}")
    print(sep)

    return {'adf_stat': adf_stat, 'adf_p': adf_p,
            'kpss_stat': kpss_stat, 'kpss_p': kpss_p,
            'adf_concl': adf_concl, 'kpss_concl': kpss_concl}

print("=" * 65)
print("  TESTS DE ESTACIONARIEDAD — BTC-USD")
print("=" * 65)
res_nivel = test_estacionariedad(btc.Price,        'Precio BTC-USD (nivel)')
res_diff1 = test_estacionariedad(btc.Price.diff(), 'Delta Precio (1a diferencia)')
res_log   = test_estacionariedad(btc.LogReturn,    'Log-Retornos')

## 2.2 Resultados e interpretación

### Serie en nivel — Precio BTC-USD

| Test | Estadístico | p-valor | Conclusión |
|------|-------------|---------|------------|
| ADF  | −1.1537 | 0.6932 | **No rechaza raíz unitaria** |
| KPSS | 7.4995  | 0.0100 | **Rechaza estacionariedad** |

Ambos tests coinciden: la serie en nivel es **no estacionaria — proceso I(1)**.

### Primera diferencia — $\Delta P_t$

| Test | Estadístico | p-valor | Conclusión |
|------|-------------|---------|------------|
| ADF  | −9.9024 | 0.0000 | **Rechaza raíz unitaria** ✓ |
| KPSS | 0.0778  | 0.1000 | **No rechaza estacionariedad** ✓ |

La primera diferencia es estacionaria → **orden de integración $d = 1$**.

In [ ]:
# ACF y PACF de la primera diferencia
serie_diff = btc.Price.diff().dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(serie_diff,  lags=40, alpha=0.05, ax=axes[0], title='ACF — Delta Precio BTC-USD')
plot_pacf(serie_diff, lags=40, alpha=0.05, ax=axes[1], title='PACF — Delta Precio BTC-USD', method='ywm')
plt.tight_layout()
plt.savefig('fig2_acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()

:::{admonition} Conclusión — Capítulo 2
:class: tip

**Orden de integración:** $d = 1$ confirmado por convergencia de ADF y KPSS.

**ACF/PACF de $\Delta P_t$:** Caída rápida sin patrones estacionales. El corte abrupto en el PACF en lag 1 sugiere un componente AR de orden bajo, consistente con los resultados de `auto_arima` que selecciona predominantemente ARIMA(1,1,0).

**Interpretación física:** BTC sigue un proceso I(1) — cada shock tiene efecto **permanente** sobre el nivel. Los incrementos (primera diferencia) son estacionarios pero exhiben heterocedasticidad condicional (clusters de volatilidad), característica que ARIMA no modela pero GARCH sí.

**Implicación para el modelado:** Se fija $d=1$ en `auto_arima` para buscar los órdenes óptimos $p$ y $q$.
:::